In [112]:
import pandas as pd
import numpy as np
import os

In [114]:
df = pd.read_csv("https://raw.githubusercontent.com/sylviacoder/Maze-Personal-Finance-Intelligence-Tracker/refs/heads/main/data/raw/personal_finance_tracker_dataset%5B1%5D.csv")

In [115]:
df.isna().sum()

date                      0
user_id                   0
monthly_income            0
monthly_expense_total     0
savings_rate              0
budget_goal               0
financial_scenario        0
credit_score              0
debt_to_income_ratio      0
loan_payment              0
investment_amount         0
subscription_services     0
emergency_fund            0
transaction_count         0
fraud_flag                0
discretionary_spending    0
essential_spending        0
income_type               0
rent_or_mortgage          0
category                  0
cash_flow_status          0
financial_advice_score    0
financial_stress_level    0
actual_savings            0
savings_goal_met          0
dtype: int64

In [118]:
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

In [120]:
df = df.drop(columns=['date'])

Date and Year Extraction Completed!

In [123]:
cat_cols = ['financial_scenario', 'income_type', 'category','cash_flow_status', 'financial_stress_level']
for col in cat_cols:
    df[col] = df[col].astype('category').cat.codes

Encoding Categorical Columns Done!

In [126]:
df.head()

,user_id,monthly_income,monthly_expense_total,savings_rate,budget_goal,financial_scenario,credit_score,debt_to_income_ratio,loan_payment,investment_amount,...,income_type,rent_or_mortgage,category,cash_flow_status,financial_advice_score,financial_stress_level,actual_savings,savings_goal_met,year,month
0,1584,3119.58,3212.07,0.38,3676.11,0,721.0,0.56,125.77,689.22,...,0,1501.65,6,2,8.3,1,0.00,0,2019,1
1,1045,3262.44,3732.81,0.10,2607.17,0,670.0,0.42,454.19,360.34,...,2,1603.17,6,2,22.6,1,0.00,0,2019,1
2,1756,2931.20,3335.58,0.15,3004.14,0,691.0,0.24,971.82,0.00,...,0,1097.82,4,2,58.8,1,0.00,0,2019,3
3,1724,3506.79,2327.59,0.17,3346.97,1,717.0,0.16,482.76,182.06,...,0,1155.64,3,2,74.5,1,1179.20,0,2019,4
4,1600,4606.87,2182.58,0.34,2670.09,0,795.0,0.25,263.74,342.78,...,2,1170.86,9,0,38.7,0,2424.29,0,2019,5


In [128]:
print("Missing values:\n", df.isnull().sum())
print("\nData types:\n", df.dtypes)
print("\nShape:", df.shape)

Missing values:
 user_id                   0
monthly_income            0
monthly_expense_total     0
savings_rate              0
budget_goal               0
financial_scenario        0
credit_score              0
debt_to_income_ratio      0
loan_payment              0
investment_amount         0
subscription_services     0
emergency_fund            0
transaction_count         0
fraud_flag                0
discretionary_spending    0
essential_spending        0
income_type               0
rent_or_mortgage          0
category                  0
cash_flow_status          0
financial_advice_score    0
financial_stress_level    0
actual_savings            0
savings_goal_met          0
year                      0
month                     0
dtype: int64

Data types:
 user_id                     int64
monthly_income            float64
monthly_expense_total     float64
savings_rate              float64
budget_goal               float64
financial_scenario           int8
credit_score            

No Missing Values

In [131]:
df['expense_income_diff'] = df['monthly_expense_total'] / df['monthly_income']
df['savings_delay_ratio'] = df['budget_goal'] - df['actual_savings']
df['debt_duress'] = df['loan_payment'] / df['monthly_income']


In [133]:
df.head()

,user_id,monthly_income,monthly_expense_total,savings_rate,budget_goal,financial_scenario,credit_score,debt_to_income_ratio,loan_payment,investment_amount,...,cash_flow_status,financial_advice_score,financial_stress_level,actual_savings,savings_goal_met,year,month,expense_income_diff,savings_delay_ratio,debt_duress
0,1584,3119.58,3212.07,0.38,3676.11,0,721.0,0.56,125.77,689.22,...,2,8.3,1,0.00,0,2019,1,1.029648,3676.11,0.040316
1,1045,3262.44,3732.81,0.10,2607.17,0,670.0,0.42,454.19,360.34,...,2,22.6,1,0.00,0,2019,1,1.144177,2607.17,0.139218
2,1756,2931.20,3335.58,0.15,3004.14,0,691.0,0.24,971.82,0.00,...,2,58.8,1,0.00,0,2019,3,1.137957,3004.14,0.331543
3,1724,3506.79,2327.59,0.17,3346.97,1,717.0,0.16,482.76,182.06,...,2,74.5,1,1179.20,0,2019,4,0.663738,2167.77,0.137664
4,1600,4606.87,2182.58,0.34,2670.09,0,795.0,0.25,263.74,342.78,...,0,38.7,0,2424.29,0,2019,5,0.473766,245.80,0.057249


In [135]:
def calculate_stress_label(row):
    score = 0
    if row['debt_to_income_ratio'] > 0.45: score += 4
    elif row['debt_to_income_ratio'] > 0.30: score += 2
    
    if row['savings_rate'] < 0.05: score += 3
    elif row['savings_rate'] < 0.15: score += 1
    
    budget_usage = row['monthly_expense_total'] / (row['budget_goal'] + 1)
    if budget_usage > 1.1: score += 3
    elif budget_usage > 0.9: score += 1

    coverage = row['emergency_fund'] / (row['monthly_expense_total'] + 1)
    if coverage < 1: score += 3
    elif coverage < 3: score += 1

    if score >= 8: return 'High'
    elif score >= 4: return 'Medium'
    else: return 'Low'

df['financial_stress_level'] = df.apply(calculate_stress_label, axis=1)


Financial Stress Label Feature Engineering
This code performs feature engineering by creating a new variable, financial_stress_level, using a rule-based scoring system. It evaluates each individual across four financial indicators: debt-to-income ratio, savings rate, budget usage, and emergency fund coverage. Each condition contributes weighted points to a cumulative stress score, where riskier financial behaviors (e.g., high debt, low savings, overspending, low emergency funds) add more points. Small constants are added in divisions to prevent errors like division by zero. Finally, the function is applied row-wise to the dataset to generate the new feature. This engineered label simplifies multiple financial metrics into a single interpretable category, making it useful for analysis or as a target variable in machine learning models.

In [138]:
df.head()

,user_id,monthly_income,monthly_expense_total,savings_rate,budget_goal,financial_scenario,credit_score,debt_to_income_ratio,loan_payment,investment_amount,...,cash_flow_status,financial_advice_score,financial_stress_level,actual_savings,savings_goal_met,year,month,expense_income_diff,savings_delay_ratio,debt_duress
0,1584,3119.58,3212.07,0.38,3676.11,0,721.0,0.56,125.77,689.22,...,2,8.3,Medium,0.00,0,2019,1,1.029648,3676.11,0.040316
1,1045,3262.44,3732.81,0.10,2607.17,0,670.0,0.42,454.19,360.34,...,2,22.6,High,0.00,0,2019,1,1.144177,2607.17,0.139218
2,1756,2931.20,3335.58,0.15,3004.14,0,691.0,0.24,971.82,0.00,...,2,58.8,Medium,0.00,0,2019,3,1.137957,3004.14,0.331543
3,1724,3506.79,2327.59,0.17,3346.97,1,717.0,0.16,482.76,182.06,...,2,74.5,Low,1179.20,0,2019,4,0.663738,2167.77,0.137664
4,1600,4606.87,2182.58,0.34,2670.09,0,795.0,0.25,263.74,342.78,...,0,38.7,Low,2424.29,0,2019,5,0.473766,245.80,0.057249


Observation: The financial stress level was left as an object to be encoded when modelling

In [141]:
mod_df = df.copy()

In [143]:
print(os.getcwd())

C:\Users\obian\OneDrive\Documents\SJ Lytix Projects\Maze


In [145]:
mod_df.to_csv(r'C:\Users\obian\OneDrive\Documents\SJ Lytix Projects\Maze\cleaned_data.csv',index=False)